# Side Effect Weighted Search

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 17.


In [ ]:
# ============================================================
# HYBRID UNIQUE-263
# CALIBRATION-ONLY WEIGHTED ENSEMBLE SEARCH
#
# Seeds: 42, 43, 44
#
# NO TRAINING.
#
# Stage A:
#   Search ensemble weights using CALIBRATION ONLY.
#
# Weight search:
#   each seed weight from 0.15 to 0.50
#   step = 0.05
#   weights sum to 1.0
#
# Equal ensemble (1/3 each) is evaluated separately.
#
# Threshold protocol is IDENTICAL:
#   Global:    0.05 -> 0.85 step 0.01
#   Per-label: 0.01 -> 0.95 step 0.01
#   support >= 3
#   prediction >= threshold
#   ties -> smallest threshold
#   method objective =
#       0.5 * Micro-F1 + 0.5 * Macro-F1
#   method tie -> GLOBAL
#
# TEST GATE:
#   Weighted calibration gain must be >= +0.0005
#   over equal hybrid.
#
# If gate FAILS:
#   TEST IS NOT LOADED / NOT EVALUATED.
#
# If gate PASSES:
#   evaluate exactly ONE selected weighted ensemble on TEST.
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

import json
import shutil
import warnings
import time

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")


# ============================================================
# 2) CONFIG
# ============================================================

SEEDS = [42, 43, 44]

N_LABELS = 263
N_CAL = 28768
N_TEST = 28768

MIN_CAL_SUPPORT = 3

GAIN_REQUIRED = 0.0005


GLOBAL_GRID = np.round(
    np.arange(
        0.05,
        0.851,
        0.01
    ),
    2
)


PER_LABEL_GRID = np.round(
    np.arange(
        0.01,
        0.951,
        0.01
    ),
    2
)


WEIGHT_GRID = np.round(
    np.arange(
        0.15,
        0.501,
        0.05
    ),
    2
)


# ============================================================
# 3) DATA PATHS
# ============================================================

STAGE2_DATA_DIR = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

TRAIN_DATA_DIR = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)


Y_CAL_PATH = (
    STAGE2_DATA_DIR /
    "Y_calibration.npy"
)

Y_TEST_PATH = (
    STAGE2_DATA_DIR /
    "Y_test.npy"
)

LABEL_MAP_PATH = (
    TRAIN_DATA_DIR /
    "label_mapping.csv"
)


# ============================================================
# 4) OUTPUT
# ============================================================

OUT_DIR = Path(
    "/kaggle/working/"
    "FINAL_UNIQUE263_HYBRID_WEIGHTED_SEARCH"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5) CHECK TARGET FILES
# ============================================================

print("=" * 120)
print("WEIGHTED HYBRID SEARCH")
print("=" * 120)


for path in [
    Y_CAL_PATH,
    Y_TEST_PATH,
    LABEL_MAP_PATH
]:

    print(
        "" if path.exists() else "",
        path
    )


if not Y_CAL_PATH.exists():
    raise FileNotFoundError(Y_CAL_PATH)

if not LABEL_MAP_PATH.exists():
    raise FileNotFoundError(LABEL_MAP_PATH)


# ============================================================
# 6) LOAD CALIBRATION TARGET ONLY
# ============================================================

print("\nLoading CALIBRATION target only...")

Y_cal = np.load(
    Y_CAL_PATH,
    mmap_mode="r"
)


label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values(
        "label_id"
    )
    .reset_index(drop=True)
)


LABELS = (
    label_map["label"]
    .astype(str)
    .tolist()
)


assert Y_cal.shape == (
    N_CAL,
    N_LABELS
)

assert len(LABELS) == N_LABELS


print(
    "Y_cal:",
    Y_cal.shape
)

print(
    "Labels:",
    len(LABELS)
)


# ============================================================
# 7) UNIQUE-263 QA
# ============================================================

expected_special = {
    "anaphylaxis": 4,
    "hyperglycemia": 4,
    "tinnitus": 3
}


print("\n" + "=" * 120)
print("CALIBRATION TARGET QA")
print("=" * 120)


for name, expected_support in expected_special.items():

    ids = (
        label_map.loc[
            label_map["label"].astype(str) == name,
            "label_id"
        ]
        .astype(int)
        .tolist()
    )

    if len(ids) != 1:

        raise RuntimeError(
            f"Could not uniquely locate {name}"
        )

    j = ids[0]

    support = int(
        np.asarray(
            Y_cal[:, j]
        ).sum()
    )

    print(
        name,
        "| ID=",
        j,
        "| calibration support=",
        support
    )

    assert support == expected_support


print(
    " Correct UNIQUE-263 calibration targets"
)


# ============================================================
# 8) FIND SEED FOLDER
# ============================================================

def find_seed_folder(seed):

    candidates = []

    for root in [
        Path("/kaggle/working"),
        Path("/kaggle/input")
    ]:

        if not root.exists():
            continue

        for p in root.rglob(
            "calibration_probabilities.npy"
        ):

            folder = p.parent

            text = str(
                folder
            ).lower()

            if (
                f"seed_{seed}"
                not in text
            ):
                continue

            if "partial" in text:
                continue

            score = 0

            if str(folder).startswith(
                "/kaggle/working"
            ):
                score += 100

            if folder.name == f"seed_{seed}":
                score += 50

            if (
                f"final_unique263_hybrid_seed_{seed}"
                in folder.name.lower()
            ):
                score += 40

            if (
                folder /
                "training_complete.json"
            ).exists():
                score += 30

            candidates.append(
                (
                    score,
                    folder
                )
            )

    if not candidates:

        raise FileNotFoundError(
            f"Could not locate Seed {seed}"
        )

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    chosen = candidates[0][1]

    print(
        f"Seed {seed}:",
        chosen
    )

    return chosen


# ============================================================
# 9) LOCATE SEEDS
# ============================================================

print("\n" + "=" * 120)
print("LOCATING SEEDS")
print("=" * 120)


seed_dirs = {
    seed: find_seed_folder(seed)
    for seed in SEEDS
}


# ============================================================
# 10) LOAD CALIBRATION PROBABILITIES ONLY
# ============================================================

print("\n" + "=" * 120)
print("LOADING CALIBRATION PROBABILITIES ONLY")
print("=" * 120)


P_cal = {}


for seed in SEEDS:

    path = (
        seed_dirs[seed] /
        "calibration_probabilities.npy"
    )

    arr = np.load(
        path
    ).astype(
        np.float32
    )

    assert arr.shape == (
        N_CAL,
        N_LABELS
    )

    assert np.isfinite(
        arr
    ).all()

    P_cal[seed] = arr

    print(
        f"Seed {seed}:",
        arr.shape
    )


print(
    "\n Test probabilities have NOT been loaded."
)


# ============================================================
# 11) FAST THRESHOLD COUNTS
# ============================================================

def threshold_counts_sorted(
    probs,
    y_true,
    thresholds
):

    probs = np.asarray(
        probs,
        dtype=np.float32
    )

    y_true = np.asarray(
        y_true,
        dtype=np.uint8
    )


    order = np.argsort(
        probs
    )


    p_sorted = probs[
        order
    ]


    y_sorted = y_true[
        order
    ]


    n = len(
        p_sorted
    )


    total_pos = int(
        y_sorted.sum()
    )


    cumulative_positive = np.cumsum(
        y_sorted[::-1],
        dtype=np.int64
    )[::-1]


    idx = np.searchsorted(
        p_sorted,
        thresholds,
        side="left"
    )


    predicted_positive = (
        n
        -
        idx
    ).astype(
        np.int64
    )


    tp = np.zeros(
        len(thresholds),
        dtype=np.int64
    )


    valid = (
        idx
        <
        n
    )


    tp[valid] = cumulative_positive[
        idx[valid]
    ]


    fp = (
        predicted_positive
        -
        tp
    )


    fn = (
        total_pos
        -
        tp
    )


    denominator = (
        2 * tp
        +
        fp
        +
        fn
    )


    f1 = np.divide(
        2 * tp,
        denominator,
        out=np.zeros(
            len(thresholds),
            dtype=np.float64
        ),
        where=(
            denominator
            >
            0
        )
    )


    return (
        tp,
        fp,
        fn,
        f1
    )


# ============================================================
# 12) COMPLETE CALIBRATION OF ONE PROBABILITY MATRIX
# ============================================================

def calibrate_probability_matrix(
    probs
):

    global_tp = np.zeros(
        len(GLOBAL_GRID),
        dtype=np.int64
    )

    global_fp = np.zeros(
        len(GLOBAL_GRID),
        dtype=np.int64
    )

    global_fn = np.zeros(
        len(GLOBAL_GRID),
        dtype=np.int64
    )

    global_macro_sum = np.zeros(
        len(GLOBAL_GRID),
        dtype=np.float64
    )


    per_label_thresholds = np.zeros(
        N_LABELS,
        dtype=np.float32
    )

    per_label_tp = np.zeros(
        N_LABELS,
        dtype=np.int64
    )

    per_label_fp = np.zeros(
        N_LABELS,
        dtype=np.int64
    )

    per_label_fn = np.zeros(
        N_LABELS,
        dtype=np.int64
    )

    per_label_f1 = np.zeros(
        N_LABELS,
        dtype=np.float64
    )


    supports = np.asarray(
        Y_cal.sum(
            axis=0
        )
    ).reshape(-1)


    # --------------------------------------------------------
    # PROCESS EACH LABEL ONCE
    # --------------------------------------------------------

    for j in range(N_LABELS):

        y = np.asarray(
            Y_cal[:, j],
            dtype=np.uint8
        )

        p = probs[:, j]


        # GLOBAL GRID
        (
            tp_g,
            fp_g,
            fn_g,
            f1_g
        ) = threshold_counts_sorted(
            p,
            y,
            GLOBAL_GRID
        )


        global_tp += tp_g
        global_fp += fp_g
        global_fn += fn_g

        global_macro_sum += f1_g


        # PER-LABEL GRID
        if int(
            supports[j]
        ) >= MIN_CAL_SUPPORT:

            (
                tp_l,
                fp_l,
                fn_l,
                f1_l
            ) = threshold_counts_sorted(
                p,
                y,
                PER_LABEL_GRID
            )


            # np.argmax returns first maximum.
            # Grid is ascending.
            # Therefore tie -> smallest threshold.
            best_idx = int(
                np.argmax(
                    f1_l
                )
            )


            per_label_thresholds[j] = (
                PER_LABEL_GRID[
                    best_idx
                ]
            )


            per_label_tp[j] = (
                tp_l[
                    best_idx
                ]
            )

            per_label_fp[j] = (
                fp_l[
                    best_idx
                ]
            )

            per_label_fn[j] = (
                fn_l[
                    best_idx
                ]
            )

            per_label_f1[j] = (
                f1_l[
                    best_idx
                ]
            )


    # ========================================================
    # GLOBAL RESULTS
    # ========================================================

    global_denominator = (
        2 * global_tp
        +
        global_fp
        +
        global_fn
    )


    global_micro = np.divide(
        2 * global_tp,
        global_denominator,
        out=np.zeros(
            len(GLOBAL_GRID),
            dtype=np.float64
        ),
        where=(
            global_denominator
            >
            0
        )
    )


    global_macro = (
        global_macro_sum
        /
        N_LABELS
    )


    global_objective = (
        0.5 * global_micro
        +
        0.5 * global_macro
    )


    best_global_idx = int(
        np.argmax(
            global_objective
        )
    )


    best_global_threshold = float(
        GLOBAL_GRID[
            best_global_idx
        ]
    )


    best_global_micro = float(
        global_micro[
            best_global_idx
        ]
    )


    best_global_macro = float(
        global_macro[
            best_global_idx
        ]
    )


    best_global_objective = float(
        global_objective[
            best_global_idx
        ]
    )


    # ========================================================
    # PER-LABEL RESULTS
    # ========================================================

    total_tp = int(
        per_label_tp.sum()
    )

    total_fp = int(
        per_label_fp.sum()
    )

    total_fn = int(
        per_label_fn.sum()
    )


    denom_micro = (
        2 * total_tp
        +
        total_fp
        +
        total_fn
    )


    per_label_micro = (
        2 * total_tp
        /
        denom_micro
        if denom_micro > 0
        else 0.0
    )


    per_label_macro = float(
        per_label_f1.mean()
    )


    per_label_objective = float(
        0.5 * per_label_micro
        +
        0.5 * per_label_macro
    )


    # ========================================================
    # SELECT METHOD
    # tie -> GLOBAL
    # ========================================================

    if (
        per_label_objective
        >
        best_global_objective
        +
        1e-15
    ):

        selected_method = "per_label"

        selected_objective = (
            per_label_objective
        )

        selected_micro = float(
            per_label_micro
        )

        selected_macro = float(
            per_label_macro
        )

        selected_thresholds = (
            per_label_thresholds.copy()
        )

    else:

        selected_method = "global"

        selected_objective = (
            best_global_objective
        )

        selected_micro = (
            best_global_micro
        )

        selected_macro = (
            best_global_macro
        )

        selected_thresholds = np.full(
            N_LABELS,
            best_global_threshold,
            dtype=np.float32
        )


    return {

        "method":
            selected_method,

        "global_threshold":
            best_global_threshold,

        "global_micro_f1":
            best_global_micro,

        "global_macro_f1":
            best_global_macro,

        "global_objective":
            best_global_objective,

        "per_label_micro_f1":
            float(
                per_label_micro
            ),

        "per_label_macro_f1":
            float(
                per_label_macro
            ),

        "per_label_objective":
            per_label_objective,

        "selected_micro_f1":
            selected_micro,

        "selected_macro_f1":
            selected_macro,

        "selected_objective":
            selected_objective,

        "thresholds":
            selected_thresholds
    }


# ============================================================
# 13) EQUAL ENSEMBLE CALIBRATION
# ============================================================

print("\n" + "=" * 120)
print("EQUAL ENSEMBLE CALIBRATION")
print("=" * 120)


equal_cal = (
    P_cal[42]
    +
    P_cal[43]
    +
    P_cal[44]
) / 3.0


equal_cal = equal_cal.astype(
    np.float32
)


equal_result = (
    calibrate_probability_matrix(
        equal_cal
    )
)


print(
    "Method:",
    equal_result[
        "method"
    ]
)

print(
    "Global threshold:",
    equal_result[
        "global_threshold"
    ]
)

print(
    "Micro-F1:",
    round(
        equal_result[
            "selected_micro_f1"
        ],
        6
    )
)

print(
    "Macro-F1:",
    round(
        equal_result[
            "selected_macro_f1"
        ],
        6
    )
)

print(
    "Objective:",
    round(
        equal_result[
            "selected_objective"
        ],
        6
    )
)


print(
    "\nPrevious run expected approximately:"
)

print(
    "Objective = 0.928388"
)


# ============================================================
# 14) GENERATE WEIGHT CANDIDATES
# ============================================================

weight_candidates = []


for w42 in WEIGHT_GRID:

    for w43 in WEIGHT_GRID:

        w44 = round(
            1.0
            -
            float(w42)
            -
            float(w43),
            2
        )


        if (
            w44
            <
            0.15
            -
            1e-9
        ):
            continue


        if (
            w44
            >
            0.50
            +
            1e-9
        ):
            continue


        # enforce 0.05 grid
        if not np.isclose(
            (
                w44
                /
                0.05
            ),
            round(
                w44
                /
                0.05
            ),
            atol=1e-8
        ):
            continue


        if not np.isclose(
            w42
            +
            w43
            +
            w44,
            1.0
        ):
            continue


        weight_candidates.append(
            (
                float(w42),
                float(w43),
                float(w44)
            )
        )


weight_candidates = sorted(
    list(
        set(
            weight_candidates
        )
    )
)


print("\n" + "=" * 120)
print("WEIGHT SEARCH SPACE")
print("=" * 120)

print(
    "Candidates:",
    len(
        weight_candidates
    )
)

print(
    "Each weight between 0.15 and 0.50"
)

print(
    "Step = 0.05"
)

print(
    "Sum = 1.00"
)


# ============================================================
# 15) CALIBRATION-ONLY WEIGHT SEARCH
# ============================================================

search_rows = []

best_weighted = None

search_start = time.time()


print("\n" + "=" * 120)
print("CALIBRATION-ONLY WEIGHT SEARCH")
print("=" * 120)


for idx, (
    w42,
    w43,
    w44
) in enumerate(
    weight_candidates,
    start=1
):


    weighted_cal = (

        w42
        *
        P_cal[42]

        +

        w43
        *
        P_cal[43]

        +

        w44
        *
        P_cal[44]

    ).astype(
        np.float32
    )


    result = (
        calibrate_probability_matrix(
            weighted_cal
        )
    )


    row = {

        "w42":
            w42,

        "w43":
            w43,

        "w44":
            w44,

        "method":
            result[
                "method"
            ],

        "global_threshold":
            result[
                "global_threshold"
            ],

        "global_objective":
            result[
                "global_objective"
            ],

        "per_label_objective":
            result[
                "per_label_objective"
            ],

        "selected_micro_f1":
            result[
                "selected_micro_f1"
            ],

        "selected_macro_f1":
            result[
                "selected_macro_f1"
            ],

        "selected_objective":
            result[
                "selected_objective"
            ]
    }


    search_rows.append(
        row
    )


    if (
        best_weighted is None
        or
        result[
            "selected_objective"
        ]
        >
        best_weighted[
            "result"
        ][
            "selected_objective"
        ]
        +
        1e-15
    ):

        best_weighted = {

            "weights":
                (
                    w42,
                    w43,
                    w44
                ),

            "result":
                result
        }


    print(
        f"{idx:02d}/{len(weight_candidates)}"
        f" | "
        f"({w42:.2f}, {w43:.2f}, {w44:.2f})"
        f" | "
        f"{result['method']}"
        f" | "
        f"Micro={result['selected_micro_f1']:.6f}"
        f" | "
        f"Macro={result['selected_macro_f1']:.6f}"
        f" | "
        f"Obj={result['selected_objective']:.6f}"
    )


search_seconds = (
    time.time()
    -
    search_start
)


# ============================================================
# 16) SAVE SEARCH TABLE
# ============================================================

search_df = pd.DataFrame(
    search_rows
).sort_values(
    "selected_objective",
    ascending=False
).reset_index(
    drop=True
)


search_df.to_csv(
    OUT_DIR /
    "weighted_calibration_search.csv",
    index=False
)


print("\n" + "=" * 120)
print("TOP CALIBRATION WEIGHTS")
print("=" * 120)


print(
    search_df.head(
        15
    ).to_string(
        index=False
    )
)


# ============================================================
# 17) BEST WEIGHTED RESULT
# ============================================================

best_w42, best_w43, best_w44 = (
    best_weighted[
        "weights"
    ]
)


best_result = (
    best_weighted[
        "result"
    ]
)


equal_objective = float(
    equal_result[
        "selected_objective"
    ]
)


weighted_objective = float(
    best_result[
        "selected_objective"
    ]
)


calibration_gain = (
    weighted_objective
    -
    equal_objective
)


print("\n" + "=" * 120)
print("CALIBRATION GATE")
print("=" * 120)


print(
    "Equal weights:",
    (
        1/3,
        1/3,
        1/3
    )
)


print(
    "Equal objective:",
    f"{equal_objective:.9f}"
)


print(
    "\nBest weighted:"
)

print(
    "w42 =",
    best_w42
)

print(
    "w43 =",
    best_w43
)

print(
    "w44 =",
    best_w44
)


print(
    "Method =",
    best_result[
        "method"
    ]
)


print(
    "Micro-F1 =",
    f"{best_result['selected_micro_f1']:.9f}"
)


print(
    "Macro-F1 =",
    f"{best_result['selected_macro_f1']:.9f}"
)


print(
    "Objective =",
    f"{weighted_objective:.9f}"
)


print(
    "\nCalibration gain:",
    f"{calibration_gain:+.9f}"
)


print(
    "Required gain:",
    f"+{GAIN_REQUIRED:.9f}"
)


GATE_PASSED = (
    calibration_gain
    >=
    GAIN_REQUIRED
)


print(
    "\nGATE:",
    " PASSED"
    if GATE_PASSED
    else
    " FAILED"
)


# ============================================================
# 18) SAVE SELECTED THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label_id":
        np.arange(
            N_LABELS
        ),

    "label":
        LABELS,

    "threshold":
        best_result[
            "thresholds"
        ]
})


threshold_df.to_csv(
    OUT_DIR /
    "best_weighted_selected_thresholds.csv",
    index=False
)


# ============================================================
# 19) SAVE CALIBRATION SUMMARY
# ============================================================

calibration_summary = {

    "equal_weights": {
        "w42": 1/3,
        "w43": 1/3,
        "w44": 1/3,
        "method":
            equal_result[
                "method"
            ],
        "micro_f1":
            float(
                equal_result[
                    "selected_micro_f1"
                ]
            ),
        "macro_f1":
            float(
                equal_result[
                    "selected_macro_f1"
                ]
            ),
        "objective":
            equal_objective
    },

    "best_weighted": {
        "w42":
            best_w42,
        "w43":
            best_w43,
        "w44":
            best_w44,
        "method":
            best_result[
                "method"
            ],
        "micro_f1":
            float(
                best_result[
                    "selected_micro_f1"
                ]
            ),
        "macro_f1":
            float(
                best_result[
                    "selected_macro_f1"
                ]
            ),
        "objective":
            weighted_objective
    },

    "calibration_gain":
        float(
            calibration_gain
        ),

    "required_gain":
        GAIN_REQUIRED,

    "gate_passed":
        bool(
            GATE_PASSED
        ),

    "test_used_during_weight_search":
        False,

    "search_seconds":
        float(
            search_seconds
        )
}


with open(
    OUT_DIR /
    "weighted_calibration_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        calibration_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 20) TEST EVALUATION FUNCTION
# ============================================================

def full_metrics(
    y_true,
    y_pred
):

    return {

        "exact_match":
            float(
                np.mean(
                    np.all(
                        y_true
                        ==
                        y_pred,
                        axis=1
                    )
                )
            ),

        "micro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),

        "micro_precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
            ),

        "micro_recall":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
            ),

        "macro_precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "macro_recall":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "hamming_loss":
            float(
                hamming_loss(
                    y_true,
                    y_pred
                )
            )
    }


# ============================================================
# 21) GATE DECISION
# ============================================================

if not GATE_PASSED:

    print("\n" + "=" * 120)
    print("STOPPING BEFORE TEST")
    print("=" * 120)

    print(
        " Weighted ensemble did not achieve "
        "the predeclared calibration gain."
    )

    print(
        "TEST probabilities were NOT loaded."
    )

    print(
        "TEST metrics were NOT calculated."
    )

    print(
        "\nKeep the equal ensemble."
    )


else:

    # ========================================================
    # 22) ONLY NOW LOAD TEST
    # ========================================================

    print("\n" + "=" * 120)
    print("GATE PASSED — LOADING TEST ONCE")
    print("=" * 120)


    if not Y_TEST_PATH.exists():

        raise FileNotFoundError(
            Y_TEST_PATH
        )


    Y_test = np.load(
        Y_TEST_PATH,
        mmap_mode="r"
    )


    assert Y_test.shape == (
        N_TEST,
        N_LABELS
    )


    P_test = {}


    for seed in SEEDS:

        test_path = (
            seed_dirs[seed] /
            "test_probabilities.npy"
        )


        if not test_path.exists():

            raise FileNotFoundError(
                test_path
            )


        arr = np.load(
            test_path
        ).astype(
            np.float32
        )


        assert arr.shape == (
            N_TEST,
            N_LABELS
        )


        P_test[seed] = arr


        print(
            f" Seed {seed} test loaded"
        )


    # ========================================================
    # 23) ONE FINAL WEIGHTED TEST
    # ========================================================

    weighted_test = (

        best_w42
        *
        P_test[42]

        +

        best_w43
        *
        P_test[43]

        +

        best_w44
        *
        P_test[44]

    ).astype(
        np.float32
    )


    thresholds = (
        best_result[
            "thresholds"
        ]
    )


    weighted_pred = (

        weighted_test

        >=

        thresholds[
            None,
            :
        ]

    ).astype(
        np.uint8
    )


    test_metrics = full_metrics(
        Y_test,
        weighted_pred
    )


    print("\n" + "=" * 120)
    print("FINAL WEIGHTED HYBRID TEST")
    print("=" * 120)


    print(
        "Weights:"
    )

    print(
        f"Seed42={best_w42:.2f}, "
        f"Seed43={best_w43:.2f}, "
        f"Seed44={best_w44:.2f}"
    )


    print(
        "Threshold method:",
        best_result[
            "method"
        ]
    )


    print(
        "Exact Match     :",
        f"{test_metrics['exact_match']*100:.4f}%"
    )

    print(
        "Micro-F1        :",
        f"{test_metrics['micro_f1']*100:.4f}%"
    )

    print(
        "Macro-F1        :",
        f"{test_metrics['macro_f1']*100:.4f}%"
    )

    print(
        "Weighted-F1     :",
        f"{test_metrics['weighted_f1']*100:.4f}%"
    )

    print(
        "Micro Precision :",
        f"{test_metrics['micro_precision']*100:.4f}%"
    )

    print(
        "Micro Recall    :",
        f"{test_metrics['micro_recall']*100:.4f}%"
    )

    print(
        "Macro Precision :",
        f"{test_metrics['macro_precision']*100:.4f}%"
    )

    print(
        "Macro Recall    :",
        f"{test_metrics['macro_recall']*100:.4f}%"
    )

    print(
        "Hamming Loss    :",
        f"{test_metrics['hamming_loss']:.8f}"
    )


    # ========================================================
    # 24) PER-LABEL TEST
    # ========================================================

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            Y_test,
            weighted_pred,
            average=None,
            zero_division=0
        )
    )


    per_label_test = pd.DataFrame({

        "label_id":
            np.arange(
                N_LABELS
            ),

        "label":
            LABELS,

        "threshold":
            thresholds,

        "test_support":
            support.astype(
                int
            ),

        "predicted_positive":
            weighted_pred.sum(
                axis=0
            ).astype(
                int
            ),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1
    })


    per_label_test.to_csv(
        OUT_DIR /
        "FINAL_weighted_test_per_label.csv",
        index=False
    )


    # ========================================================
    # 25) SAVE TEST METRICS
    # ========================================================

    test_result_row = {
        "w42":
            best_w42,
        "w43":
            best_w43,
        "w44":
            best_w44,
        "threshold_method":
            best_result[
                "method"
            ],
        "calibration_objective":
            weighted_objective,
        **test_metrics
    }


    pd.DataFrame(
        [
            test_result_row
        ]
    ).to_csv(
        OUT_DIR /
        "FINAL_weighted_test_metrics.csv",
        index=False
    )


    # ========================================================
    # 26) REPLACEMENT LABELS
    # ========================================================

    print("\n" + "=" * 120)
    print("REPLACEMENT LABELS")
    print("=" * 120)


    special = (
        per_label_test[
            per_label_test[
                "label"
            ].isin(
                [
                    "anaphylaxis",
                    "hyperglycemia",
                    "tinnitus"
                ]
            )
        ]
    )


    print(
        special.to_string(
            index=False
        )
    )


# ============================================================
# 27) ZIP OUTPUT
# ============================================================

ZIP_PATH = shutil.make_archive(
    "/kaggle/working/"
    "FINAL_UNIQUE263_HYBRID_WEIGHTED_SEARCH",
    "zip",
    root_dir=OUT_DIR
)


print("\n" + "=" * 120)
print("WEIGHTED SEARCH COMPLETE")
print("=" * 120)


print(
    "Output folder:"
)

print(
    OUT_DIR
)


print(
    "\nZIP:"
)

print(
    ZIP_PATH
)


if GATE_PASSED:

    print(
        "\n Calibration gate PASSED."
    )

    print(
        " One final weighted TEST evaluation performed."
    )

else:

    print(
        "\n Calibration gate FAILED."
    )

    print(
        " Test remained unevaluated in this experiment."
    )


print(
    "\n NO TRAINING was performed."
)